# Week 3 - 적대적 학습: pix2pix GAN

## 오늘 할 것
1. L1 손실만 쓰면 왜 결과가 뿌옇게 나오는지 눈으로 확인한다.
2. 두 번째 신경망인 **판별자(Discriminator)** 를 만들고, hinge 손실을 숫자로 이해한다.
3. GAN을 직접 학습한다. 처음부터 학습하는 방법과, W2 체크포인트를 이어받아 미세조정하는 방법 둘 다 다룬다.
4. 복원 결과를 원본과 비교하고, GAN이 무엇을 바꾸는지 정직하게 평가한다.

> 이번 주는 **Bentheimer 사암**으로 실습한다. 균질한 암석이라 구조가 단순해서 복원이 잘 되는 편이라, 모델이 슬라이스를 얼마나 잘 되살리는지 눈으로 보기 좋다. 이웃 거리는 **k=2**(측정한 슬라이스 사이에 2칸씩 비어 있는 상황)로 잡는다.

## 0. 환경 준비

In [ ]:
import sys
from pathlib import Path

# helpers(dr_utils.py, model_utils.py)는 같은 폴더(다운로드 zip)나 ../helpers(저장소)에 있다.
for _cand in [Path('.'), Path('..') / 'helpers']:
    if (_cand / 'dr_utils.py').exists():
        sys.path.insert(0, str(_cand.resolve())); break

import numpy as np
import matplotlib.pyplot as plt
import torch

from dr_utils import (
    load_volume, porosity, predict_linear_k, eval_targets,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)
from model_utils import (
    UNetMini, count_parameters, train_quick, evaluate_model, load_ckpt,
    # --- W3에서 새로 쓰는 것 ---
    PatchDiscriminatorMini, ssim_loss, d_hinge_loss, g_hinge_loss,
    train_gan, predict_continuous, save_gan_ckpt, load_gan_ckpt,
)
setup_plot_style()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print(f'PyTorch {torch.__version__}, device={DEVICE}')

## 1. W2 복습, 그리고 오늘의 문제

W2에서 만든 UNet은 선형 보간을 크게 이겼다. 그런데 결과를 자세히 보면 아직 아쉬운 점이 있다. 이진화(0/1)하기 전의 출력을 그대로 보면, pore 경계가 **회색으로 번져** 있다.

왜 그럴까? 이웃 두 장만으로는 가운데 픽셀이 pore인지 solid인지 확실하지 않은 자리가 생깁니다. 이럴 때 L1 손실은 위험을 줄이려고 그냥 **중간값(회색)** 을 내놓는다. 어느 쪽도 아닌, 진짜 암석에는 없는 값이다. 이걸 "평균의 함정"이라고 부른다.

In [ ]:
DATA = next((p for p in [Path('data'), Path('..') / 'data'] if (p / 'Bentheimer_256.bin').exists()), Path('data'))
vol = load_volume(DATA / 'Bentheimer_256.bin')
K = 2   # 이웃 거리 (측정 슬라이스 사이 간격)

# 선형 보간 baseline. 우리가 이겨야 할 대상이다.
m_lin = eval_targets(predict_linear_k(vol, K), vol, K)
print(f'선형 보간 (k={K})   |Δφ|={m_lin["dphi_pp"]:.2f}%p   SSIM={m_lin["ssim"]:.3f}')

# L1만 쓴 모델을 하나 학습한다 (판별자 없이 재구성 손실만).
#   train_gan(lambda_gan=0) 이면 GAN 없이 L1+SSIM만 학습한다.
print('\nL1 위주 모델 학습 중...')
G_l1, _, _ = train_gan(vol, k=K, preset='fast', lambda_gan=0.0, w_ssim=0.0,
                       epochs=22, warmup=0, d_base=16, device=DEVICE, verbose=True)

### 회색 번짐을 눈으로 확인

In [ ]:
# 고정 위치의 patch 하나로 관찰한다.
z, c0, c1 = 128, 64, 192
before, after, target = vol[z-K, c0:c1, c0:c1], vol[z+K, c0:c1, c0:c1], vol[z, c0:c1, c0:c1]

cont_l1 = predict_continuous(G_l1, before, after, device=DEVICE)
grey = lambda c: ((c > 0.2) & (c < 0.8)).mean()   # 회색(불확실) 픽셀 비율

fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
ax[0].imshow(target, cmap='gray_r', vmin=0, vmax=1); ax[0].set_title('원본 (정답)')
ax[1].imshow(cont_l1, cmap='gray_r', vmin=0, vmax=1); ax[1].set_title(f'L1 출력 · 회색 {grey(cont_l1)*100:.0f}%')
row = int(np.argmax(np.abs(np.diff(target, axis=1)).sum(axis=1)))
for a in ax[:2]: a.axhline(row, color=ORANGE, ls='--', lw=1.3); a.set_xticks([]); a.set_yticks([])
ax[2].plot(target[row], color=NAVY, lw=2.4, label='정답 (계단)')
ax[2].plot(cont_l1[row], color=RED, lw=2.2, label='L1 (완만 = 회색)')
ax[2].axhline(0.5, color=GRAY, ls=':', lw=1); ax[2].legend(fontsize=9); ax[2].set_title('가로 단면')
plt.tight_layout(); plt.show()
print('경계에서 값이 0.5 근처로 완만하게 넘어간다. 이게 회색 번짐이다.')

## 2. 판별자 만들기

GAN의 핵심은 두 번째 신경망인 **판별자 D** 다. 슬라이스를 받아서 "이게 진짜 암석 단면이야?" 하고 점수를 매긴다. 우리 판별자는 세 가지 장치를 쓴다.

- **조건부(conditional)**: 이웃 슬라이스도 같이 넣는다. 그래서 "사실적이면서 이웃과도 맞는가"를 본다.
- **PatchGAN**: 이미지를 조각(patch)으로 나눠서 조각마다 점수를 낸다. 국소적인 경계와 텍스처에 민감해진다.
- **spectral norm**: 판별자의 힘에 상한을 걸어 학습을 안정시킨다.

한 가지 더. 우리 생성자(mini UNet)는 아주 작기 때문에, 판별자를 너무 크게 만들면 판별자가 일방적으로 이겨서 학습이 망가진다. 그래서 판별자 폭을 `d_base=16` 정도로 작게 잡는다.

In [ ]:
D = PatchDiscriminatorMini(cond_ch=2, base=16)
print('D 파라미터:', f'{count_parameters(D):,}')

# 입력: 조건(2채널) + 판정 대상(1채널) -> patch별 점수 map
cond = torch.randn(1, 2, 64, 64)
y    = torch.rand(1, 1, 64, 64)
score = D(cond, y)
print('입력 조건', tuple(cond.shape), '+ 대상', tuple(y.shape), '-> 점수 map', tuple(score.shape))
print('점수 map의 각 칸이 한 patch에 대한 진짜(+)/가짜(-) 판정이다.')

## 3. 적대적 손실 (hinge)

**판별자 D** 는 진짜에 +1 이상, 가짜에 -1 이하를 주려고 한다.

$$\mathcal{L}_D = \mathbb{E}\,[\max(0,\,1-D(y))] + \mathbb{E}\,[\max(0,\,1+D(G(x)))]$$

**생성자 G** 는 판별자를 속이려고, 즉 점수를 올리려고 한다.

$$\mathcal{L}_G^{\text{adv}} = -\,\mathbb{E}\,[D(G(x))]$$

"확실히 맞힌 것에는 벌점 0, 애매한 것에만 벌점"이라는 규칙이다. 숫자로 직접 확인해 본다.

In [ ]:
real_score = torch.tensor([0.8, 1.5, -0.2])   # 진짜에 매긴 점수
fake_score = torch.tensor([-0.4, 0.3, -1.5])  # 가짜에 매긴 점수
d_pen = torch.relu(1 - real_score).mean() + torch.relu(1 + fake_score).mean()
g_adv = -fake_score.mean()
print('D 벌점 (진짜는 +1 이상, 가짜는 -1 이하면 0):', round(float(d_pen), 3))
print('G 손실 (-D(fake), 작을수록 잘 속인 것):', round(float(g_adv), 3))
print('real=1.5, fake=-1.5 는 확실히 맞혀서 벌점 0. real=0.8, fake=0.3 은 애매해서 벌점이 생긴다.')

## 4. GAN 학습 (경로 1: 처음부터)

`train_gan` 이 warmup, hinge, spectral norm을 다 처리한다. 처음 몇 epoch은 재구성 손실(L1, SSIM)만으로 몸을 풀고(λ=0), 그 다음에 판별자를 붙인다. `snapshot` 을 주면 학습 중간의 출력을 저장해서 나중에 변화를 볼 수 있다.

In [ ]:
G, D, hist = train_gan(vol, k=K, preset='fast', lambda_gan=0.1, w_ssim=0.3,
                       epochs=30, warmup=8, d_base=16, d_every=1, lambda_decay=0.3,
                       snapshot=(before, after), snapshot_every=3,
                       device=DEVICE, verbose=True)
save_gan_ckpt(G, D, 'w3_gan_mini.pth', meta={'g_base': 16, 'd_base': 16, 'k': K, 'domain': 'Bentheimer'})
print('\n체크포인트 저장: w3_gan_mini.pth')

### 학습 곡선 읽는 법

두 신경망이 서로 당기는 학습이라, 곡선을 보면 상태를 알 수 있다.

- 왼쪽: 재구성 손실(L1, SSIM). GAN이 켜진 뒤에도 계속 내려가면, 사실감을 더하면서도 정확도를 해치지 않는다는 뜻이다.
- 오른쪽: 판별자 손실(D)과 생성자 적대 손실(G). **D 손실이 0에 딱 붙지 않고 어느 정도 값을 유지**하면 건강한 균형이다. D 손실이 0으로 떨어지면 판별자가 일방적으로 이긴 것이라, 생성자가 배우지 못한다.

In [ ]:
warmup = 8
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3))
ax[0].plot(hist['G_l1'], color=NAVY, lw=2, label='L1')
ax[0].plot(hist['G_ssim'], color=ORANGE, lw=2, label='SSIM 손실')
ax[0].axvline(warmup, color=GRAY, ls='--'); ax[0].text(warmup+0.3, max(hist['G_l1'])*0.85, 'GAN 시작', color=GRAY)
ax[0].set_title('생성자 재구성 손실'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.2)

ax[1].plot(hist['D_loss'], color=RED, lw=2, label='판별자 D')
ax[1].plot(hist['G_gan'], color=GREEN, lw=2, label='생성자 적대 G')
ax[1].axhline(0, color=GRAY, lw=0.8); ax[1].axvline(warmup, color=GRAY, ls='--')
ax[1].axhspan(0.2, 0.8, color=GREEN, alpha=0.06)
ax[1].set_title('적대적 손실 (D와 G의 줄다리기)'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.2)
plt.tight_layout(); plt.show()
d_post = np.array(hist['D_loss'])[np.array(hist['lam']) > 0]
print(f'GAN 구간 D 손실 평균 {d_post.mean():.2f} (0.2~0.8 사이면 건강한 균형). 0에 안 붙었으면 정상이다.')

## 5. 복원 결과를 원본과 비교

이제 학습한 GAN 모델로 빠진 슬라이스를 복원해서 원본과 나란히 본다. 오른쪽 끝은 원본과 복원의 차이(밝을수록 틀린 부분)이다.

In [ ]:
res_gan = evaluate_model(G, vol, k=K, device=DEVICE)
lin_recon = predict_linear_k(vol, K)
zc = 129   # 복원된(합성된) 슬라이스 하나
fig, ax = plt.subplots(1, 4, figsize=(15, 4.2))
panels = [('원본 (정답)', vol[zc], NAVY), (f'선형 보간', lin_recon[zc], RED),
          ('UNet + GAN', res_gan['recon'][zc], ORANGE),
          ('|원본 - GAN|', np.abs(vol[zc] - res_gan['recon'][zc]), None)]
for a, (t, im, col) in zip(ax, panels):
    a.imshow(im, cmap='hot' if col is None else 'gray_r', vmin=0, vmax=1)
    a.set_title(t, color=col if col else 'k', fontsize=13, fontweight='bold'); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print(f'선형 |Δφ|={m_lin["dphi_pp"]:.2f}%p  ->  GAN |Δφ|={res_gan["dphi_pp"]:.2f}%p, SSIM={res_gan["ssim"]:.3f}')
print('균질한 Bentheimer라 복원이 원본에 꽤 가깝게 나온다.')

### GAN이 바꾸는 것: 연속 출력의 선명도

In [ ]:
cont_gan = predict_continuous(G, before, after, device=DEVICE)
fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
for a, im, t, col in [(ax[0], target, '원본', NAVY), (ax[1], cont_l1, f'L1 · 회색 {grey(cont_l1)*100:.0f}%', RED),
                      (ax[2], cont_gan, f'GAN · 회색 {grey(cont_gan)*100:.0f}%', GREEN)]:
    a.imshow(im, cmap='gray_r', vmin=0, vmax=1); a.set_title(t, color=col); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print(f'회색(불확실) 비율: L1 {grey(cont_l1)*100:.0f}% -> GAN {grey(cont_gan)*100:.0f}%. GAN 쪽이 더 또렷하다.')

## 6. 정직한 평가: 선형 vs L1 vs GAN

여기서 이번 주의 중요한 포인트가 나온다. 세 방법을 같은 조건에서 비교해 본다.

In [ ]:
# 공정 비교: L1+SSIM 모델(GAN 없음)과 GAN 모델을 같은 설정으로.
G_fair, _, _ = train_gan(vol, k=K, preset='fast', lambda_gan=0.0, w_ssim=0.3,
                         epochs=30, warmup=8, d_base=16, device=DEVICE, verbose=False)
res_fair = evaluate_model(G_fair, vol, k=K, device=DEVICE)
print(f'{"방법":<16}{"|Δφ|(%p)":>10}{"SSIM":>9}')
print(f'{"선형 보간":<15}{m_lin["dphi_pp"]:>10.2f}{m_lin["ssim"]:>9.3f}')
print(f'{"UNet (GAN 없음)":<13}{res_fair["dphi_pp"]:>10.2f}{res_fair["ssim"]:>9.3f}')
print(f'{"UNet + GAN":<15}{res_gan["dphi_pp"]:>10.2f}{res_gan["ssim"]:>9.3f}')

> **관찰**: 선형 보간에서 딥러닝으로 넘어가면 오차가 크게 줄어든다(11.75 → 1.2 아래로). GAN을 켜면 여기서는 |Δφ|가 한 번 더 내려갔다. 다만 이 차이는 크지 않고, 학습의 무작위성에 따라 비슷하게 나오거나 살짝 반대로 뒤집힐 수도 있다. 여러 번 돌려 보면 감이 온다.
>
> 확실한 것은 **GAN이 연속 출력을 더 선명하게** 만든다는 점이다(위에서 본 회색 비율). GAN의 목표는 픽셀 오차를 더 줄이는 게 아니라 결과를 진짜처럼 만드는 것이라, 그 효과가 선명도에서 먼저 보인다. 큰 모델로 가면 투과율 같은 물성 지표에서 이득이 뚜렷해진다(강의 슬라이드의 LBM 결과). 픽셀 지표 하나로는 이 차이를 다 잡지 못해서, 우리는 S2나 투과율 같은 물리 지표도 함께 본다(W5).

## 7. 경로 2: W2 체크포인트 이어받아 미세조정

처음부터 학습하는 대신, W2에서 만든 UNet을 생성자 초기값으로 이어받을 수도 있다. 이미 기본기가 있어서 warmup을 짧게 줄일 수 있다. (여기서는 W2 모델을 즉석에서 하나 만들어 보여준다. 실제로는 W2에서 저장한 `unet_mini_*.pth` 를 `load_ckpt` 로 불러오면 된다.)

In [ ]:
G0, _ = train_quick(vol, k=K, preset='fast', device=DEVICE, verbose=False)
print('W2 UNet 준비 완료. 이어서 GAN 미세조정...')
G_ft, D_ft, _ = train_gan(vol, k=K, generator=G0, lambda_gan=0.1, w_ssim=0.3,
                          epochs=15, warmup=2, d_base=16, device=DEVICE, verbose=True)
cont_ft = predict_continuous(G_ft, before, after, device=DEVICE)
print(f'\n미세조정 결과 회색 비율: {grey(cont_ft)*100:.0f}%. W2 이어받기는 warmup을 짧게 줄여도 된다.')

## 8. 심화: λ로 흐림과 환각 사이 조절

adversarial 가중치 **λ** 가 균형을 정한다. 너무 작으면 L1과 같아서 뿌옇고, 너무 크면 이웃과 안 맞는 구조를 지어내는 환각이 생긴다. λ를 몇 개 바꿔서 선명도(회색 비율)와 |Δφ|가 어떻게 움직이는지 본다.

In [ ]:
for lam in [0.0, 0.1, 0.3]:
    Gi, _, _ = train_gan(vol, k=K, preset='fast', lambda_gan=lam, w_ssim=0.3,
                         epochs=26, warmup=6, d_base=16, device=DEVICE, verbose=False)
    ci = predict_continuous(Gi, before, after, device=DEVICE)
    ri = evaluate_model(Gi, vol, k=K, device=DEVICE)
    print(f'λ={lam:<4}  회색 {grey(ci)*100:4.0f}%   |Δφ| {ri["dphi_pp"]:.2f}%p   SSIM {ri["ssim"]:.3f}')
print('\nGAN 학습은 예민하다. λ를 바꾸면 결과가 꽤 흔들린다.')
print('특히 λ를 크게 넣으면 |Δφ|가 오히려 나빠지기 쉽다(위 표).')
print('그래서 실전에서는 λ를 작게(0.1 근처) 넣어 재구성 손실이 주도하게 한다. GAN은 살짝 얹는 양념이다.')

## 9. 정리, 그리고 다음 주

**오늘 배운 것**
- L1 손실은 애매한 자리에서 회색을 낸다. "평균의 함정"이다.
- GAN은 생성자와 판별자를 경쟁시켜, 회색 대신 진짜 같은 구조를 만들게 한다.
- 조건부, PatchGAN, spectral norm, hinge 손실, 그리고 작은 λ가 실전 레시피다.
- 판별자가 너무 세면(D 손실이 0에 붙으면) 학습이 망가진다. 판별자를 작게(`d_base=16`) 잡아 균형을 맞췄다.
- GAN은 출력을 더 선명하게 만들고, 여기서는 |Δφ|도 함께 좋아졌다. 다만 픽셀 지표만으로는 GAN의 이득을 다 못 보기 때문에, 큰 모델에서는 투과율 같은 물성 지표로 확인한다.

**다음 주 (W4)**: 다른 아키텍처(Transformer 기반, 3D)를 같은 문제에 적용해 오늘의 GAN과 비교한다. 학습한 체크포인트(`w3_gan_mini.pth`)를 보관하라.

**탐구 과제**(핸드아웃 §5): GAN 유무 비교, λ sweep, warmup·spectral norm 실험, "숫자는 비슷한데 구조가 다른" 사례 찾기.